# 2 - Weak signals: which detector catches the outage?

**Question.** Ticket volume per category per day. Something breaks - a
migration, a certificate, a bad patch - and the tickets arrive before anyone
files an incident. Which detector raises that, how early, and how often does it
cry wolf?

| Approach | What it is |
|---|---|
| `fixed` | `mean + 3 sigma` over a trailing window |
| `ewma` | Seasonal robust baseline + Poisson tail + EWMA of residuals (shipped) |
| `prophet` | Forecast, alert above the prediction interval |

**All three arms are mandatory.** Prophet used to sit behind an optional extra
and was skipped when absent; it is a hard dependency now, so the comparison
cannot quietly become a two-way one.

### Two things decide this, and neither is accuracy

**False alarms per 100 series-days.** Precision alone is misleading: the number
of alerts depends on how long an anomaly lasts, so a detector firing for six
days on one real outage looks precise. The normalised rate is what an on-call
team experiences.

**Detection delay.** A detector that finds the outage on day 5 is a post-mortem
tool. The whole claim of weak-signal detection is the *slow ramp* - the one no
threshold catches, because no single day is unusual.

### The design decision is upstream of all three

**Series grain.** Per category, per CI, per site? It sets the false-alarm rate
before a line of code runs. Section 6 sweeps it, and it moves the numbers more
than the choice of detector.

```bash
uv sync
uv run itsm data build      # dataset + the injected-anomaly manifest
```

### Runtime

Prophet dominates: one model fit per series, and the default 1000 posterior
samples are what make each fit slow. `UNCERTAINTY_SAMPLES = 200` cuts that
roughly fourfold for an interval that moves in the third decimal. The other two
detectors are vectorised and take under a second.

In [1]:
import json
import logging
import time

import numpy as np
import pandas as pd

from itsm.config import get_settings
from itsm.data.loading import load_anomaly_manifest
from itsm.models.weak_signal import (
    Alert,
    DetectionReport,
    WeakSignalDetector,
    build_count_series,
    evaluate_detections,
)
from itsm.pipelines.anonymise import load_raw_tickets

pd.set_option('display.width', 160)

settings = get_settings()
GRAIN = 'category'
TOLERANCE_DAYS = 2        # grace period after a window before an alert is a false alarm
UNCERTAINTY_SAMPLES = 200  # Prophet posterior draws; 1000 is the default and 5x slower

## 1. Series and ground truth

The generator injects anomalies of known shape - sudden spikes and slow ramps -
and records each window. Without that manifest, precision and recall here would
be opinions.

Gaps are filled with zeros on purpose: **a day with no tickets is information,
not a missing value**.

In [2]:
tickets = load_raw_tickets(settings)
counts = build_count_series(tickets, by=GRAIN)
manifest = list(load_anomaly_manifest(settings))
n_series_days = int(counts.shape[0] * counts.shape[1])

if not manifest:
    raise RuntimeError('no anomaly manifest - run `uv run itsm data build` first')

print(f'{counts.shape[1]} series x {counts.shape[0]} days = {n_series_days} series-days')
print(f'{len(manifest)} injected anomalies')
pd.DataFrame(manifest)

12 series x 365 days = 4380 series-days
4 injected anomalies


,category,kind,start,end,peak_multiplier,label
0,Network & VPN,spike,2026-02-26,2026-02-28,5.0,VPN concentrator outage
1,Email & Collaboration,ramp,2026-04-29,2026-05-23,3.2,Progressive mailbox migration degradation
2,Business Applications,level_shift,2026-07-04,2026-09-15,1.8,ERP release introducing a regression
3,Security,spike,2026-08-09,2026-08-10,7.0,Phishing campaign


In [3]:
# Volume per series decides whether a series is scoreable at all. Under about
# one ticket a day, any sigma or Poisson rule fires on 2 tickets - the detector
# is not at fault, the grain is.
volume = counts.mean().sort_values(ascending=False).to_frame('mean tickets/day')
volume['days at zero'] = (counts == 0).sum()
volume.round(2)

,mean tickets/day,days at zero
category,,
Access & Accounts,4.18,68
Network & VPN,2.51,95
Email & Collaboration,2.25,96
Business Applications,2.23,95
Hardware & Workstation,2.16,95
Software & Licensing,1.81,109
Security,1.32,134
Printing,1.25,131
Telephony & Meeting Rooms,1.15,144


## 2. The two challengers

Both return the same `DetectionReport` as the shipped detector, so one scorer
measures all three against one manifest.

**The fixed threshold is the real baseline.** It is what a service desk already
has, it takes one line of SQL, and anything that cannot clearly beat it does
not deserve a dependency.

In [4]:
def fixed_threshold(counts, sigmas=3.0, window=56, min_observed=5, min_history=21):
    '''Alert when today exceeds mean + sigmas * std of the trailing window.

    Vectorised with `rolling(...).shift(1)`: the shift is what keeps it causal,
    excluding today from its own baseline. The naive version recomputed the
    mean and std per day inside a Python loop, which is O(days x window) per
    series and was the slowest non-Prophet cell in the notebook.

    No z-score or tail probability exists here - that absence is the point of
    the comparison, not an omission.
    '''
    alerts = []
    for name in counts.columns:
        series = counts[name].astype(float)
        rolling = series.rolling(window, min_periods=min_history).agg(['mean', 'std']).shift(1)
        threshold = rolling['mean'] + sigmas * rolling['std']
        fires = (series >= min_observed) & series.gt(threshold)
        for period in series.index[fires]:
            observed, mean, limit = series[period], rolling['mean'][period], threshold[period]
            severity = float(observed / limit) if limit else float(observed)
            alerts.append(
                Alert(
                    series=str(name),
                    period=period.date(),
                    kind='spike',
                    observed=int(observed),
                    expected=float(mean),
                    robust_z=float('nan'),
                    poisson_p=float('nan'),
                    ewma=float('nan'),
                    severity=round(severity, 3),
                )
            )
    return DetectionReport(alerts=alerts, diagnostics=pd.DataFrame())

In [5]:
def prophet_detect(counts, interval_width=0.99, min_observed=5, min_history=60):
    '''Fit trend + weekly seasonality per series, alert above yhat_upper.

    In-sample on purpose: Prophet is fitted on the whole series and residuals
    are read off the same data. That is optimistic - the anomaly is in the
    training set, pulling the fit towards it - and it is how people actually
    use Prophet for anomaly detection. The other two arms are causal; section 7
    repeats the caveat where the conclusion is written.

    Speed: `uncertainty_samples` is the cost driver. The default 1000 posterior
    draws are computed for every row of every series; 200 gives an interval
    that differs in the third decimal for a fifth of the time. `yearly` is off
    because a year of data cannot support it, and that is a fit fewer to solve.
    '''
    from prophet import Prophet

    # Prophet logs a Stan banner per fit; with one fit per series that is noise.
    logging.getLogger('cmdstanpy').setLevel(logging.WARNING)
    logging.getLogger('prophet').setLevel(logging.WARNING)

    alerts = []
    for name in counts.columns:
        series = counts[name]
        if len(series) < min_history:
            continue
        frame = pd.DataFrame({'ds': pd.to_datetime(series.index), 'y': series.to_numpy(float)})
        model = Prophet(
            growth='flat',
            interval_width=interval_width,
            weekly_seasonality=True,
            yearly_seasonality=False,
            daily_seasonality=False,
            uncertainty_samples=UNCERTAINTY_SAMPLES,
        )
        model.fit(frame)
        forecast = model.predict(frame)

        observed = frame['y'].to_numpy()
        expected = forecast['yhat'].to_numpy()
        upper = forecast['yhat_upper'].to_numpy()
        fires = np.nonzero((observed >= min_observed) & (observed > upper))[0]
        for position in fires:
            width = max(float(upper[position] - expected[position]), 1e-9)
            alerts.append(
                Alert(
                    series=str(name),
                    period=frame['ds'][position].date(),
                    kind='spike',
                    observed=int(observed[position]),
                    expected=float(expected[position]),
                    robust_z=float('nan'),
                    poisson_p=float('nan'),
                    ewma=float('nan'),
                    severity=round(float((observed[position] - expected[position]) / width), 3),
                )
            )
    return DetectionReport(alerts=alerts, diagnostics=pd.DataFrame())

In [6]:
DETECTORS = {
    'fixed': fixed_threshold,
    'ewma': lambda counts: WeakSignalDetector().detect(counts),
    'prophet': prophet_detect,
}

# Written down before the run. A hypothesis recorded afterwards is a story.
EXPECTED = {
    'fixed': 'catches loud spikes, misses every slow ramp, noisy on low-volume series',
    'ewma': 'catches both shapes; the EWMA arm is what finds the ramp',
    'prophet': 'good seasonality, slow, optimistic because it fits the anomaly',
}

reports, scores = {}, {}
for name, detect in DETECTORS.items():
    started = time.perf_counter()
    report = detect(counts)
    elapsed = time.perf_counter() - started

    metrics = evaluate_detections(
        report, manifest, tolerance_days=TOLERANCE_DAYS, n_series_days=n_series_days
    )
    reports[name] = report
    scores[name] = {
        'anomalies caught': str(metrics['detected']) + '/' + str(metrics['n_anomalies']),
        'recall': metrics['recall'],
        'false alarms / 100 series-days': metrics.get('false_alarms_per_100_series_days'),
        'mean detection delay (days)': metrics['mean_detection_delay_days'],
        'alerts raised': metrics['n_alerts'],
        'seconds': round(elapsed, 1),
        'expected': EXPECTED[name],
        '_metrics': metrics,
    }
    print(f'{name:9s} {metrics["n_alerts"]:4d} alerts  {elapsed:6.1f}s')

fixed       46 alerts     0.0s
ewma        93 alerts     2.5s


Matplotlib is building the font cache; this may take a moment.
Importing plotly failed. Interactive plots will not work.
07:45:16 - cmdstanpy - INFO - Chain [1] start processing
07:45:16 - cmdstanpy - INFO - Chain [1] done processing
07:45:16 - cmdstanpy - INFO - Chain [1] start processing
07:45:16 - cmdstanpy - INFO - Chain [1] done processing
07:45:16 - cmdstanpy - INFO - Chain [1] start processing
07:45:16 - cmdstanpy - INFO - Chain [1] done processing
07:45:16 - cmdstanpy - INFO - Chain [1] start processing
07:45:16 - cmdstanpy - INFO - Chain [1] done processing
07:45:17 - cmdstanpy - INFO - Chain [1] start processing
07:45:17 - cmdstanpy - INFO - Chain [1] done processing
07:45:17 - cmdstanpy - INFO - Chain [1] start processing
07:45:17 - cmdstanpy - INFO - Chain [1] done processing
07:45:17 - cmdstanpy - INFO - Chain [1] start processing
07:45:17 - cmdstanpy - INFO - Chain [1] done processing
07:45:17 - cmdstanpy - INFO - Chain [1] start processing
07:45:17 - cmdstanpy - INFO - C

prophet     82 alerts    33.4s


## 3. Headline

Read `recall` and `false alarms / 100 series-days` **together**. Either alone is
trivially gamed: alert on everything for perfect recall, alert on nothing for a
perfect false-alarm rate. The question is what recall costs in pages at 3am.

In [7]:
comparison = pd.DataFrame(
    {
        name: {key: value for key, value in score.items() if not key.startswith('_')}
        for name, score in scores.items()
    }
)
comparison

,fixed,ewma,prophet
anomalies caught,4/4,4/4,4/4
recall,1.0,1.0,1.0
false alarms / 100 series-days,0.89,1.712,1.484
mean detection delay (days),18.75,19.0,5.75
alerts raised,46,93,82
seconds,0.0,2.5,33.4
expected,"catches loud spikes, misses every slow ramp, n...",catches both shapes; the EWMA arm is what find...,"good seasonality, slow, optimistic because it ..."


## 4. Spikes vs slow ramps

The aggregate hides the finding. A sudden spike is easy - every detector should
get it. **The slow ramp is the whole argument**: a drift where no single day
crosses any threshold, and where only an accumulating statistic can fire.

If the two columns below are equal, the EWMA arm is not earning its complexity
and the fixed threshold is the honest answer.

In [8]:
def by_kind(metrics):
    buckets = {}
    for entry in metrics['per_anomaly']:
        bucket = buckets.setdefault(entry['kind'], {'detected': 0, 'total': 0, 'delays': []})
        bucket['total'] += 1
        if entry['detected']:
            bucket['detected'] += 1
            if entry['detection_delay_days'] is not None:
                bucket['delays'].append(entry['detection_delay_days'])
    summary = {}
    for kind, bucket in sorted(buckets.items()):
        summary[kind + ': caught'] = str(bucket['detected']) + '/' + str(bucket['total'])
        delays = bucket['delays']
        summary[kind + ': mean delay'] = round(float(np.mean(delays)), 1) if delays else None
    return summary


pd.DataFrame({name: by_kind(score['_metrics']) for name, score in scores.items()})

,fixed,ewma,prophet
level_shift: caught,1/1,1/1,1/1
level_shift: mean delay,62.0,62.0,10.0
ramp: caught,1/1,1/1,1/1
ramp: mean delay,12.0,12.0,12.0
spike: caught,2/2,2/2,2/2
spike: mean delay,0.5,1.0,0.5


## 5. What the alerts look like

A ranked list is the deliverable - an on-call team reads the top few, not all
of them - so what matters is whether the real anomalies are *at the top*, not
merely present somewhere.

In [9]:
for name, report in reports.items():
    frame = report.to_frame()
    print('===', name, '-', len(frame), 'alerts ===')
    if not frame.empty:
        display(frame.head(5))

=== fixed - 46 alerts ===


,series,period,kind,observed,expected,robust_z,poisson_p,ewma,severity
38,Security,2026-08-10,spike,11,1.160714,NaN,NaN,NaN,2.168
19,Hardware & Workstation,2026-06-22,spike,13,2.142857,NaN,NaN,NaN,1.694
41,Storage & File Shares,2026-01-13,spike,6,0.803571,NaN,NaN,NaN,1.680
17,Email & Collaboration,2026-09-07,spike,10,1.500000,NaN,NaN,NaN,1.646
43,Storage & File Shares,2026-09-07,spike,5,0.750000,NaN,NaN,NaN,1.504


=== ewma - 93 alerts ===


,series,period,kind,observed,expected,robust_z,poisson_p,ewma,severity
39,Email & Collaboration,2026-05-22,emerging,7,2.33,3.055,0.010068,2.842,8.526
38,Email & Collaboration,2026-05-21,spike,11,3.00,3.597,0.000292,2.751,8.253
18,Business Applications,2026-09-04,spike,12,2.00,6.745,0.000001,2.597,7.791
10,Business Applications,2025-10-24,spike,6,1.00,5.000,0.000594,2.583,7.749
24,Business Applications,2026-09-14,spike,12,4.00,4.000,0.000915,2.409,7.227


=== prophet - 82 alerts ===


,series,period,kind,observed,expected,robust_z,poisson_p,ewma,severity
14,Business Applications,2026-09-04,spike,12,2.500011,NaN,NaN,NaN,2.613
67,Security,2026-08-10,spike,11,2.320717,NaN,NaN,NaN,2.568
74,Software & Licensing,2026-06-15,spike,9,2.962162,NaN,NaN,NaN,2.264
24,Email & Collaboration,2026-03-20,spike,10,2.557656,NaN,NaN,NaN,2.209
35,Hardware & Workstation,2026-06-22,spike,13,3.603785,NaN,NaN,NaN,2.122


## 6. The grain sweep - the decision upstream of the detector

Same detector, different series definition. This usually moves the false-alarm
rate further than any threshold does.

The `all (one series)` row is expected to score a recall of 0: at that grain an
alert cannot be attributed to a category, so the scorer cannot match it to a
window. That is the coarse end of the trade-off made visible, not a bug.

*Only the EWMA detector is swept - Prophet would mean refitting every series at
every grain, which is minutes for a table that answers a question about the
grain rather than about Prophet.*

In [10]:
candidates = [None, 'category', 'subcategory', 'assignment_group']
grains = [g for g in candidates if g is None or g in set(tickets.columns)]

sweep = []
for grain in grains:
    grain_counts = build_count_series(tickets, by=grain)
    grain_days = int(grain_counts.shape[0] * grain_counts.shape[1])
    report = WeakSignalDetector().detect(grain_counts)
    metrics = evaluate_detections(
        report, manifest, tolerance_days=TOLERANCE_DAYS, n_series_days=grain_days
    )
    sweep.append(
        {
            'grain': grain or 'all (one series)',
            'series': grain_counts.shape[1],
            'mean tickets/day/series': round(float(grain_counts.mean().mean()), 2),
            'recall': metrics['recall'],
            'false alarms / 100 series-days': metrics.get('false_alarms_per_100_series_days'),
            'alerts': metrics['n_alerts'],
        }
    )

pd.DataFrame(sweep).set_index('grain')

,series,mean tickets/day/series,recall,false alarms / 100 series-days,alerts
grain,,,,,
all (one series),1,21.64,0.0,5.753,21
category,12,1.80,1.0,1.712,93
subcategory,48,0.45,0.0,0.263,46
assignment_group,11,1.97,0.0,2.092,84


## 7. Conclusion

*Fill in from the run. The structure of the argument:*

**What ships.** The detector that catches both anomaly shapes at a false-alarm
rate an on-call rota tolerates - and if the fixed threshold matches it, the
fixed threshold wins: no dependency, and anyone can explain it at 3am.

**Prophet's number is optimistic by construction.** It is fitted on the whole
series including the anomaly; the other two only ever see the past. If it does
not clearly beat them *with* that advantage, it loses. It is also two orders of
magnitude slower per run, which matters for a job that reruns hourly.

**Preconditions for production:**

* 6-8 weeks of history per series before a threshold means anything;
* enough volume per series - see section 1;
* alert ground truth defined **before** go-live, or precision is unmeasurable
  on real data the way it is measurable here;
* someone on call. An alert nobody routes is not an alert.

**Threat to validity.** The anomalies are injected with known shapes and
magnitudes. Real degradations are messier, overlap, and arrive with the
seasonality shifted by a public holiday nobody encoded.

In [11]:
settings.ensure_dirs()
path = settings.reports_dir / 'experiment_02_weak_signal.json'
path.write_text(
    json.dumps(
        {
            'grain': GRAIN,
            'n_series_days': n_series_days,
            'n_anomalies': len(manifest),
            'prophet_uncertainty_samples': UNCERTAINTY_SAMPLES,
            'headline': comparison.to_dict(),
            'grain_sweep': sweep,
        },
        indent=2,
        default=str,
    ),
    encoding='utf-8',
)
print('written to', path)

written to reports/experiment_02_weak_signal.json
